# KGIN clean Colab notebook

这是一份全新的、从头开始的 KGIN 训练 notebook。按顺序运行，不要跳着跑。

## 0. 先确认运行时

在 Colab 里先把运行时切到 **GPU**：

`Runtime -> Change runtime type -> Hardware accelerator -> GPU`

In [3]:
import torch, sys
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("请先切换到 GPU 运行时，再继续。")


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 1. 基础路径设置

这里把你现在已有的数据目录和新代码目录分开。

In [4]:
from pathlib import Path

# 你的现有数据目录（已经有 train.txt / test.txt / kg_final.txt 等）
OLD_DATASET_DIR = Path("/content/Knowledge_Graph_based_Intent_Network/data/mydata")

# 全新的代码仓库目录
REPO_DIR = Path("/content/KGIN_code")

# 新仓库里的数据目录
NEW_DATASET_DIR = REPO_DIR / "data" / "mydata"

# 导出 embedding 的输出目录
OUTPUT_DIR = NEW_DATASET_DIR / "output"

print("OLD_DATASET_DIR =", OLD_DATASET_DIR)
print("REPO_DIR =", REPO_DIR)
print("NEW_DATASET_DIR =", NEW_DATASET_DIR)


OLD_DATASET_DIR = /content/Knowledge_Graph_based_Intent_Network/data/mydata
REPO_DIR = /content/KGIN_code
NEW_DATASET_DIR = /content/KGIN_code/data/mydata


## 2. 重新 clone 一份干净的 KGIN 代码

In [5]:
%cd /content
!rm -rf /content/KGIN_code
!git clone https://github.com/huangtinglin/Knowledge_Graph_based_Intent_Network.git KGIN_code
%cd /content/KGIN_code
!pwd
!ls -lah
!find . -maxdepth 2 -type f | sort | head -80


/content
Cloning into 'KGIN_code'...
remote: Enumerating objects: 67, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 67 (delta 16), reused 54 (delta 16), pack-reused 7 (from 1)
Receiving objects: 100% (67/67), 46.51 MiB | 31.56 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/KGIN_code
/content/KGIN_code
total 44K
drwxr-xr-x 7 root root 4.0K Mar 31 05:39 .
drwxr-xr-x 1 root root 4.0K Mar 31 05:39 ..
drwxr-xr-x 5 root root 4.0K Mar 31 05:39 data
drwxr-xr-x 8 root root 4.0K Mar 31 05:39 .git
-rw-r--r-- 1 root root 5.0K Mar 31 05:39 main.py
drwxr-xr-x 2 root root 4.0K Mar 31 05:39 modules
-rw-r--r-- 1 root root 6.5K Mar 31 05:39 README.md
drwxr-xr-x 2 root root 4.0K Mar 31 05:39 training_log
drwxr-xr-x 2 root root 4.0K Mar 31 05:39 utils
./.git/config
./.git/description
./.git/HEAD
./.git/index
./.git/packed-refs
./main.py
./modules/KGIN.py
./README.md
./training_log/Alibaba-iFashion.txt
./training_log/Amazon-B

## 3. 把你已有的数据复制到新仓库里

In [6]:
!mkdir -p /content/KGIN_code/data/mydata
!cp /content/Knowledge_Graph_based_Intent_Network/data/mydata/* /content/KGIN_code/data/mydata/
!ls -lah /content/KGIN_code/data/mydata


cp: cannot stat '/content/Knowledge_Graph_based_Intent_Network/data/mydata/*': No such file or directory
total 8.0K
drwxr-xr-x 2 root root 4.0K Mar 31 05:39 .
drwxr-xr-x 6 root root 4.0K Mar 31 05:39 ..


## 4. 检查关键文件是否齐全

In [9]:
required = [
    "train.txt",
    "test.txt",
    "kg_final.txt",
    "user_list.txt",
    "item_list.txt",
    "entity_list.txt",
    "relation_list.txt",
    "item_id_map.json",
    "stats.json",
]
missing = [x for x in required if not (NEW_DATASET_DIR / x).exists()]
print("missing =", missing)
assert not missing, f"缺少文件: {missing}"
print("所有关键文件齐全。")


missing = []
所有关键文件齐全。


## 5. 安装最基础依赖

这一步尽量少装包，不要再跑旧 notebook 里那种大规模卸载/重装。

In [18]:
pip install torch-scatter torch-sparse torch-cluster torch-geometric -f https://data.pyg.org/whl/torch-2.10.0+cu128.html

Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 129.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 114.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 173.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.2 MB/s eta 0:00:00


In [10]:
!pip -q install numpy scipy pandas networkx scikit-learn tqdm
import torch, numpy, scipy, pandas
print("Torch:", torch.__version__)
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("CUDA:", torch.cuda.is_available())


Torch: 2.10.0+cu128
NumPy: 2.0.2
SciPy: 1.16.3
CUDA: True


## 6. 找训练入口文件

不同仓库版本入口文件位置可能不同，这里先自动查。

In [11]:
%cd /content/KGIN_code
!find . -maxdepth 3 -type f | egrep "main.py|train.py|parser.py|KGIN.py" | sort


/content/KGIN_code
./main.py
./modules/KGIN.py
./utils/parser.py


## 7. 开始训练

默认先尝试 `main.py`。如果这里报找不到文件，就看上一格输出，把 `main.py` 改成真实入口。

In [12]:
mkdir -p /content/KGIN_code/weights

In [13]:
import json
from pathlib import Path

data_dir = Path("/content/KGIN_code/data/mydata")

# 读取映射
item_map = json.loads((data_dir / "item_id_map.json").read_text(encoding="utf-8"))

user_map = {}
with open(data_dir / "user_list.txt", "r", encoding="utf-8") as f:
    next(f)  # skip header
    for line in f:
        org_id, remap_id = line.strip().split()
        user_map[int(org_id)] = int(remap_id)

def remap_grouped_txt(input_path, output_path):
    new_lines = []
    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            org_user = int(parts[0])
            org_items = [int(x) for x in parts[1:]]

            if org_user not in user_map:
                continue

            remap_user = user_map[org_user]
            remap_items = [item_map[str(i)] for i in org_items if str(i) in item_map]

            # 去重，保留顺序
            seen = set()
            remap_items = [x for x in remap_items if not (x in seen or seen.add(x))]

            if remap_items:
                new_lines.append(
                    " ".join([str(remap_user)] + [str(i) for i in remap_items])
                )

    with open(output_path, "w", encoding="utf-8") as f:
        for row in new_lines:
            f.write(row + "\n")

remap_grouped_txt(data_dir / "train.txt", data_dir / "train_remap.txt")
remap_grouped_txt(data_dir / "test.txt", data_dir / "test_remap.txt")

print("saved:", data_dir / "train_remap.txt")
print("saved:", data_dir / "test_remap.txt")

saved: /content/KGIN_code/data/mydata/train_remap.txt
saved: /content/KGIN_code/data/mydata/test_remap.txt


In [14]:
!mv /content/KGIN_code/data/mydata/train_remap.txt /content/KGIN_code/data/mydata/train.txt
!mv /content/KGIN_code/data/mydata/test_remap.txt /content/KGIN_code/data/mydata/test.txt

In [15]:
# 看 train/test 里最大 item id 是否已经变成 <= 3528
max_item = -1
for fn in ["/content/KGIN_code/data/mydata/train.txt", "/content/KGIN_code/data/mydata/test.txt"]:
    with open(fn, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) > 1:
                items = list(map(int, parts[1:]))
                max_item = max(max_item, max(items))
print("max item id =", max_item)

max item id = 3528


In [16]:
import fileinput
import sys

file_path = '/content/KGIN_code/utils/metrics.py'
old_line = '    r = np.asfarray(r)[:k]'
new_line = '    r = np.asarray(r, dtype=float)[:k]'

with fileinput.FileInput(file_path, inplace=True, encoding='utf-8') as file:
    for line in file:
        # Ensure we only replace the exact line to avoid unintended changes
        if line.strip() == old_line.strip():
            print(new_line)
        else:
            print(line, end='')

print(f'已更新文件: {file_path}')

# 验证修改
with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()
    if new_line.strip() in content:
        print('metrics.py 文件已成功更新。')
    else:
        print('metrics.py 文件更新失败，请手动检查。')

已更新文件: /content/KGIN_code/utils/metrics.py
metrics.py 文件已成功更新。


In [19]:
%cd /content/KGIN_code
!rm -f /content/KGIN_code/weights/model_mydata.ckpt

!python main.py \
  --dataset mydata \
  --data_path ./data/ \
  --dim 64 \
  --lr 0.0001 \
  --sim_regularity 0.0001 \
  --batch_size 1024 \
  --node_dropout True \
  --node_dropout_rate 0.5 \
  --mess_dropout True \
  --mess_dropout_rate 0.1 \
  --gpu_id 0 \
  --context_hops 3 \
  --epoch 50 \
  --save 1

/content/KGIN_code
reading train and test user-item set ...
combinating train_cf and kg data ...
building the graph ...
Begin to load interaction triples ...
100% 563206/563206 [00:00<00:00, 844539.19it/s]

Begin to load knowledge graph triples ...
100% 40366/40366 [00:00<00:00, 689802.83it/s]
building the adj mat ...
Begin to build sparse relation matrix ...
100% 7/7 [00:00<00:00, 61.19it/s]
start training ...
using time 14.8415, training loss at epoch 0: 225.8379, cor: 2019.674561
+-------+-------------------+--------------------+--------------------+----------------------------------------------------------+----------------------------------------------------------+----------------------------------------------------------+----------------------------------------------------------+
| Epoch |   training time   |    tesing time     |        Loss        |                          recall                          |                           ndcg                           |               

## 8. 查看训练后生成的权重文件

In [20]:
%cd /content/KGIN_code
!find . -type f | egrep "ckpt|pth|pt" | sort
!ls -lah ./weights || true


/content/KGIN_code
./.git/description
./weights/model_mydata.ckpt
total 4.1M
drwxr-xr-x 2 root root 4.0K Mar 31 05:42 .
drwxr-xr-x 8 root root 4.0K Mar 31 05:41 ..
-rw-r--r-- 1 root root 4.1M Mar 31 05:53 model_mydata.ckpt


## 9. 导出 KGIN movie embeddings

这里直接调用你已经上传好的 `export_kgin_embeddings.py`。

In [21]:
!ls -lah /content/KGIN_code/data/mydata
!find /content -name "export_kgin_embeddings.py"

total 3.4M
drwxr-xr-x 3 root root 4.0K Mar 31 05:41 .
drwxr-xr-x 6 root root 4.0K Mar 31 05:39 ..
-rw-r--r-- 1 root root 313K Mar 31 05:40 entity_list.txt
-rw-r--r-- 1 root root  12K Mar 31 05:40 export_for_kgat_kgin.py
drwxr-xr-x 2 root root 4.0K Mar 31 05:41 .ipynb_checkpoints
-rw-r--r-- 1 root root  57K Mar 31 05:40 item_id_map.json
-rw-r--r-- 1 root root  77K Mar 31 05:40 item_list.txt
-rw-r--r-- 1 root root 252K Mar 31 05:40 kg_final.txt
-rw-r--r-- 1 root root   64 Mar 31 05:40 relation_list.txt
-rw-r--r-- 1 root root 1.7K Mar 31 05:40 results_test.json
-rw-r--r-- 1 root root  295 Mar 31 05:40 stats.json
-rw-r--r-- 1 root root  57K Mar 31 05:40 test.txt
-rw-r--r-- 1 root root 2.6M Mar 31 05:40 train.txt
-rw-r--r-- 1 root root  63K Mar 31 05:40 user_list.txt
/content/KGIN_code/export_kgin_embeddings.py


In [22]:
!python /content/KGIN_code/export_kgin_embeddings.py \
  --repo-root /content/KGIN_code \
  --data-path /content/KGIN_code/data \
  --dataset mydata \
  --checkpoint /content/KGIN_code/weights/model_mydata.ckpt \
  --item-map /content/KGIN_code/data/mydata/item_id_map.json \
  --output-dir /content/KGIN_code/data/mydata/output \
  --gpu-id 0

reading train and test user-item set ...
combinating train_cf and kg data ...
building the graph ...
Begin to load interaction triples ...
100% 563206/563206 [00:00<00:00, 759498.38it/s]

Begin to load knowledge graph triples ...
100% 40366/40366 [00:00<00:00, 667905.15it/s]
building the adj mat ...
Begin to build sparse relation matrix ...
100% 7/7 [00:00<00:00, 60.19it/s]
[OK] Saved: /content/KGIN_code/data/mydata/output/kgin_movie_embeddings.npy
[OK] Saved: /content/KGIN_code/data/mydata/output/kgin_movie_id_map.json
[Info] shape=(3529, 64)


## 10. 检查导出结果

In [23]:
!ls -lah /content/KGIN_code/data/mydata/output


total 948K
drwxr-xr-x 2 root root 4.0K Mar 31 06:00 .
drwxr-xr-x 4 root root 4.0K Mar 31 06:00 ..
-rw-r--r-- 1 root root 883K Mar 31 06:00 kgin_movie_embeddings.npy
-rw-r--r-- 1 root root  54K Mar 31 06:00 kgin_movie_id_map.json


## 11. 如果第 9 步里 checkpoint 路径不对

先运行第 8 步，找到真实文件名，再把第 9 步里的：

`/content/KGIN_code/weights/model_mydata.ckpt`

改成第 8 步查到的真实路径。